# SOLUSDT order hedging check

Assumptions for hedging success:
- Match open and hedge orders by `strategy_id`.
- Required hedge qty = open `filled_qty` (not the original order qty).
- Hedged ok when sum of hedge `filled_qty` >= required qty (within a small tolerance).
- Rows with open `filled_qty == 0` are excluded from the unhedged list.


In [6]:
import pandas as pd

engine_used = "pyarrow"
try:
    df = pd.read_parquet("FILUSDT_order.parquet", engine="pyarrow")
except Exception:
    engine_used = "fastparquet"
    df = pd.read_parquet("FILUSDT_order.parquet", engine="fastparquet")

print(f"engine={engine_used}, rows={len(df)}, cols={df.shape[1]}")

df.head()


engine=fastparquet, rows=5781, cols=22


,symbol,order_id,client_order_id,side,order_type,price,quantity,status,trading_venue,strategy_id,...,order_side,create_ts,opening_bid0,opening_ask0,hedging_bid0,hedging_ask0,price_offset,update_ts,local_ts,filled_qty
0,FIL-USDT-SWAP,3156524743043047426,1855644365448282113,BUY,LIMIT,1.274,1568.0,CANCELED,OkexFutures,432050872,...,open,1.766574e+15,1.276,1.277,1.277,1.278,0.0015,1766574226311000000,1766574168016000000,0.0
1,FIL-USDT-SWAP,3156524743043047425,1855642660346265601,BUY,LIMIT,1.274,1568.0,CANCELED,OkexFutures,432050475,...,open,1.766574e+15,1.276,1.277,1.277,1.278,0.0009,1766574226311000000,1766574168016000000,0.0
2,FIL-USDT-SWAP,3156524743043047424,1855640684661309441,BUY,LIMIT,1.275,1568.0,CANCELED,OkexFutures,432050015,...,open,1.766574e+15,1.276,1.277,1.277,1.278,0.0005,1766574226311000000,1766574168016000000,0.0
3,FIL-USDT-SWAP,3156524742975938560,1855637695364071425,BUY,LIMIT,1.275,1566.0,CANCELED,OkexFutures,432049319,...,open,1.766574e+15,1.276,1.277,1.277,1.278,0.0001,1766574226309000000,1766574168014000000,0.0
4,FIL-USDT-SWAP,3156524742908829697,1855645224441741313,BUY,LIMIT,1.273,1570.0,CANCELED,OkexFutures,432051072,...,open,1.766574e+15,1.276,1.277,1.277,1.278,0.0020,1766574226315000000,1766574168012000000,0.0


In [7]:
open_df = df[df["order_side"] == "open"].copy()
hedge_df = df[df["order_side"] == "hedge"].copy()

open_df = open_df.rename(
    columns={
        "order_id": "open_order_id",
        "client_order_id": "open_client_order_id",
        "status": "open_status",
        "side": "open_side",
        "quantity": "open_qty",
        "filled_qty": "open_filled_qty",
        "create_ts": "open_create_ts",
        "update_ts": "open_update_ts",
        "local_ts": "open_local_ts",
    }
)

open_df["open_local_ts_utc"] = pd.to_datetime(
    open_df["open_local_ts"], unit="ns", utc=True
)


open_cols = [
    "symbol",
    "open_order_id",
    "open_client_order_id",
    "open_side",
    "order_type",
    "price",
    "open_qty",
    "open_status",
    "trading_venue",
    "strategy_id",
    "opening_venue",
    "hedging_venue",
    "open_create_ts",
    "open_update_ts",
    "open_local_ts",
    "open_local_ts_utc",
    "open_filled_qty",
]

open_df = open_df[open_cols].set_index("strategy_id")

hedge_agg = hedge_df.groupby("strategy_id").agg(
    hedge_order_count=("order_id", "size"),
    hedge_filled_count=("status", lambda s: (s == "FILLED").sum()),
    hedge_canceled_count=("status", lambda s: (s == "CANCELED").sum()),
    hedge_filled_qty_sum=("filled_qty", "sum"),
)

summary = open_df.join(hedge_agg, how="left")

for col in ["hedge_order_count", "hedge_filled_count", "hedge_canceled_count"]:
    summary[col] = summary[col].fillna(0).astype(int)
summary["hedge_filled_qty_sum"] = summary["hedge_filled_qty_sum"].fillna(0.0)

summary.head()


,symbol,open_order_id,open_client_order_id,open_side,order_type,price,open_qty,open_status,trading_venue,opening_venue,hedging_venue,open_create_ts,open_update_ts,open_local_ts,open_local_ts_utc,open_filled_qty,hedge_order_count,hedge_filled_count,hedge_canceled_count,hedge_filled_qty_sum
strategy_id,,,,,,,,,,,,,,,,,,,,
432050872,FIL-USDT-SWAP,3156524743043047426,1855644365448282113,BUY,LIMIT,1.274,1568.0,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766574e+15,1766574226311000000,1766574168016000000,2025-12-24 11:02:48.016000+00:00,0.0,0,0,0,0.0
432050475,FIL-USDT-SWAP,3156524743043047425,1855642660346265601,BUY,LIMIT,1.274,1568.0,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766574e+15,1766574226311000000,1766574168016000000,2025-12-24 11:02:48.016000+00:00,0.0,0,0,0,0.0
432050015,FIL-USDT-SWAP,3156524743043047424,1855640684661309441,BUY,LIMIT,1.275,1568.0,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766574e+15,1766574226311000000,1766574168016000000,2025-12-24 11:02:48.016000+00:00,0.0,0,0,0,0.0
432049319,FIL-USDT-SWAP,3156524742975938560,1855637695364071425,BUY,LIMIT,1.275,1566.0,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766574e+15,1766574226309000000,1766574168014000000,2025-12-24 11:02:48.014000+00:00,0.0,0,0,0,0.0
432051072,FIL-USDT-SWAP,3156524742908829697,1855645224441741313,BUY,LIMIT,1.273,1570.0,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766574e+15,1766574226315000000,1766574168012000000,2025-12-24 11:02:48.012000+00:00,0.0,0,0,0,0.0


In [8]:
okx_contract_multiplier = {
    "FIL-USDT-SWAP": 0.1,
    "DOT-USDT-SWAP": 1.0,
}

summary["open_contract_multiplier"] = 1.0
okx_mask = summary["trading_venue"] == "OkexFutures"
summary.loc[okx_mask, "open_contract_multiplier"] = (
    summary.loc[okx_mask, "symbol"].map(okx_contract_multiplier).fillna(1.0)
)

summary["need_hedge_qty"] = summary["open_filled_qty"] * summary["open_contract_multiplier"]

tol = 1e-9
summary["hedged_ok"] = summary["hedge_filled_qty_sum"] + tol >= summary["need_hedge_qty"]

need_hedge = summary[summary["need_hedge_qty"] > 0]
unhedged = need_hedge[~need_hedge["hedged_ok"]].sort_values("open_create_ts")

print(f"open rows: {len(open_df)}")
print(f"need hedge rows: {len(need_hedge)}")
print(f"unhedged rows: {len(unhedged)}")

unhedged


open rows: 3777
need hedge rows: 494
unhedged rows: 15


,symbol,open_order_id,open_client_order_id,open_side,order_type,price,open_qty,open_status,trading_venue,opening_venue,...,open_local_ts,open_local_ts_utc,open_filled_qty,hedge_order_count,hedge_filled_count,hedge_canceled_count,hedge_filled_qty_sum,open_contract_multiplier,need_hedge_qty,hedged_ok
strategy_id,,,,,,,,,,,,,,,,,,,,,
647755855,FIL-USDT-SWAP,3147164493718478848,2782090213017518081,SELL,LIMIT,1.328,752.0,FILLED,OkexFutures,OkexFutures,...,1766295210848000000,2025-12-21 05:33:30.848000+00:00,752.0,0,0,0,0.0,0.1,75.2,False
647755251,FIL-USDT-SWAP,3147164493617815554,2782087618857271297,SELL,LIMIT,1.328,752.0,FILLED,OkexFutures,OkexFutures,...,1766295210845000000,2025-12-21 05:33:30.845000+00:00,752.0,0,0,0,0.0,0.1,75.2,False
647756074,FIL-USDT-SWAP,3147164493550706688,2782091153615355905,SELL,LIMIT,1.328,752.0,FILLED,OkexFutures,OkexFutures,...,1766295210843000000,2025-12-21 05:33:30.843000+00:00,752.0,0,0,0,0.0,0.1,75.2,False
647755617,FIL-USDT-SWAP,3147164493483597824,2782089190815301633,SELL,LIMIT,1.328,752.0,FILLED,OkexFutures,OkexFutures,...,1766295210841000000,2025-12-21 05:33:30.841000+00:00,752.0,0,0,0,0.0,0.1,75.2,False
652755572,FIL-USDT-SWAP,3147164661423529984,2803563834021773313,SELL,LIMIT,1.328,752.0,FILLED,OkexFutures,OkexFutures,...,1766295215846000000,2025-12-21 05:33:35.846000+00:00,752.0,0,0,0,0.0,0.1,75.2,False
652755922,FIL-USDT-SWAP,3147164661255757824,2803565337260326913,SELL,LIMIT,1.328,752.0,FILLED,OkexFutures,OkexFutures,...,1766295215841000000,2025-12-21 05:33:35.841000+00:00,752.0,0,0,0,0.0,0.1,75.2,False
346086842,FIL-USDT-SWAP,3148739638379667456,1486431667965919233,SELL,LIMIT,1.263,1583.0,FILLED,OkexFutures,OkexFutures,...,1766342153817000000,2025-12-21 18:35:53.817000+00:00,1583.0,20,0,20,0.0,0.1,158.3,False
346087261,FIL-USDT-SWAP,3148739638245449728,1486433467557216257,SELL,LIMIT,1.263,1583.0,FILLED,OkexFutures,OkexFutures,...,1766342153813000000,2025-12-21 18:35:53.813000+00:00,1583.0,20,0,20,0.0,0.1,158.3,False
2160661,FIL-USDT-SWAP,3152186862547755008,9279968332742657,SELL,LIMIT,1.300,1538.0,FILLED,OkexFutures,OkexFutures,...,1766444889103000000,2025-12-22 23:08:09.103000+00:00,1538.0,5,1,4,73.9,0.1,153.8,False


In [9]:
def show_strategy(strategy_id: int):
    return df[df["strategy_id"] == strategy_id].sort_values(
        ["order_side", "create_ts", "order_id"]
    )

# Example: show_strategy(1870436056)
